# 🏀 NBA Game Outcome Prediction

**Predicting whether the home team wins an NBA game using only pre-game information.**

This project builds a machine-learning pipeline that predicts NBA game outcomes from historical
team performance, rest, injuries, travel, and playoff seeding — using **only data knowable before tip-off**
(no data leakage).

- **Data:** ~8,600 NBA games (2016–2022)
- **Models:** Logistic Regression, Random Forest, Gradient Boosting + Ensemble
- **Result:** **62% accuracy** vs a **56% home-court baseline** (+6 points), evaluated on held-out recent games
- **Key finding:** Conference seed difference and recent form are the strongest predictors; box-score
  features plateau around 62% — closing the gap to Vegas (~70%) requires betting-market and play-by-play data.

---

## 1. Setup & Data Loading

We use a games dataset plus injury logs, player box scores (for star-weighting injuries),
and team metadata (for travel distance).

In [ ]:
import pandas as pd, numpy as np
from collections import defaultdict, deque
import warnings; warnings.filterwarnings('ignore')

games = pd.read_csv('games.csv')
print('Raw games:', games.shape)
games.head()

## 2. Data Cleaning

Drop the small number of rows with missing values and sort chronologically — sorting is essential
so that every feature can be computed using **only past games**, never future ones.

In [ ]:
games = games.dropna().sort_values('GAME_DATE_EST').reset_index(drop=True)
print('Clean games:', len(games))

# Baseline: how often does the home team win?
print('Home win rate (baseline):', round(games['HOME_TEAM_WINS'].mean()*100,1),'%')

## 3. Avoiding Data Leakage — the Core Principle

A game's final box-score stats (points, FG%, etc.) reveal the result, so using them would be **cheating**.
Instead, every feature below is built from a team's **history coming into the game**:

- Win % and recent (last-10) form *before* this game
- Days of rest / back-to-backs (from game dates)
- Star-weighted injury impact (who's out × how much they score)
- Travel distance (from previous game location)
- Home/road splits and conference seed

The golden rule in the loops below: **record the feature first, then update history with the result.**

## 4. Feature Engineering

### 4a. Win % and recent form (last 10 games)

In [ ]:
wins, played = defaultdict(int), defaultdict(int)
last10 = defaultdict(lambda: deque(maxlen=10))
hw_pct, aw_pct, h_form, a_form = [], [], [], []

for g in games.itertuples():
    h, a = g.HOME_TEAM_ID, g.VISITOR_TEAM_ID
    hw_pct.append(wins[h]/played[h] if played[h] else 0.5)
    aw_pct.append(wins[a]/played[a] if played[a] else 0.5)
    h_form.append(sum(last10[h])/len(last10[h]) if last10[h] else 0.5)
    a_form.append(sum(last10[a])/len(last10[a]) if last10[a] else 0.5)
    res = int(g.HOME_TEAM_WINS)          # update AFTER recording
    played[h]+=1; played[a]+=1
    if res: wins[h]+=1
    else:    wins[a]+=1
    last10[h].append(res); last10[a].append(1-res)

games['home_winpct'], games['away_winpct'] = hw_pct, aw_pct
games['home_form10'], games['away_form10'] = h_form, a_form

### 4b. Rest days, injuries (star-weighted), travel, seeding

These follow the same no-leakage pattern. Injuries are weighted by each player's season scoring
average (a star being out matters far more than a benchwarmer). Seeding ranks each team within its
conference *as of that date*. Full code in the project scripts.

In [ ]:
# Difference features — models learn the GAP between teams better than absolutes
games['winpct_diff'] = games['home_winpct'] - games['away_winpct']
games['form_diff']   = games['home_form10'] - games['away_form10']
# (rest_diff, injury_diff, travel_diff, split_diff, seed_diff built the same way)
print('Feature engineering complete.')

## 5. Model Training — Honest Time-Based Split

We train on the **older 80%** of games and test on the **most recent 20%**. We never use a random
split, because predicting the past from the future is itself a form of leakage.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

df = pd.read_csv('games_model.csv').fillna(0)   # full engineered feature set
features = ['winpct_diff','form_diff','rest_diff','injury_diff','travel_diff',
            'home_form10','away_form10','late_season','h2h_home_winrate','split_diff',
            'home_home_winpct','away_road_winpct','seed_diff','home_seed','away_seed','home_locked_in']
X, y = df[features], df['HOME_TEAM_WINS']
split = int(len(df)*0.8)
Xtr,Xte,ytr,yte = X[:split],X[split:],y[:split],y[split:]
print('Train:',len(Xtr),'| Test:',len(Xte),'(most recent games)')

In [ ]:
sc = StandardScaler().fit(Xtr)
lr = LogisticRegression(C=0.1,max_iter=3000).fit(sc.transform(Xtr),ytr)
rf = RandomForestClassifier(n_estimators=400,max_depth=7,min_samples_leaf=20,random_state=42).fit(Xtr,ytr)
gb = GradientBoostingClassifier(n_estimators=300,max_depth=3,learning_rate=0.03,random_state=42).fit(Xtr,ytr)

for name,p in [('Logistic',lr.predict(sc.transform(Xte))),('Random Forest',rf.predict(Xte)),('Gradient Boost',gb.predict(Xte))]:
    print(f'{name:16s} {accuracy_score(yte,p)*100:.1f}%')

# Ensemble: average the three models' probabilities
proba=(lr.predict_proba(sc.transform(Xte))[:,1]+rf.predict_proba(Xte)[:,1]+gb.predict_proba(Xte)[:,1])/3
pe=(proba>=0.5).astype(int)
print(f'{"ENSEMBLE":16s} {accuracy_score(yte,pe)*100:.1f}%  (baseline {(yte==1).mean()*100:.1f}%)')

## 6. Evaluation & Feature Importance

In [ ]:
print('Confusion matrix:\n', confusion_matrix(yte,pe))
imp=sorted(zip(features,rf.feature_importances_),key=lambda x:-x[1])
print('\nTop predictors:')
for f,v in imp[:6]: print(f'  {f:18s} {v*100:.1f}%')

## 7. Results & Conclusions

| Model | Accuracy |
|---|---|
| Baseline (always pick home) | 56.1% |
| Logistic Regression | 61.9% |
| Random Forest | 61.5% |
| Gradient Boosting | 61.9% |
| **Ensemble** | **62.0%** |

**Top predictors:** conference seed difference, recent (last-10) form, and home/road splits.
Star-weighted injuries and travel contribute smaller but real signal.

**Honest ceiling:** Three different algorithms and an ensemble all plateau near 62%, which indicates
the limit is the *information in box scores*, not the model. Professional bookmakers reach ~70% using
data unavailable here — live injury severity, betting-market movement, and play-by-play lineup data.

### Future work
- **Betting-market features** (point spreads) — highest accuracy gain, but reduces interpretability
- **Play-by-play / lineup data** — true on-court impact, large engineering effort
- **Standings-based motivation** — full "locked-in seed" modeling (this project tested a simple proxy)
- **Player-level matchups** — positional advantages not captured by team aggregates